# 02 FX Volatility Regime Prediction

**Portfolio module:** FX Market Intelligence / Flex DA role  
**Business goal:** predict whether the next trading day is likely to be a low, normal, or high volatility FX day.

This is intentionally framed as an **operational signal**, not an exact exchange-rate forecast.  
For an FX fintech product team, volatility regime prediction can support:

- transaction monitoring readiness
- rate-change communication
- support staffing awareness
- lifecycle campaign timing
- product education during volatile markets

Main model layer: **scikit-learn**.  
Optional experiment layer: **PyTorch / TensorFlow** MLP classifiers.

## Latest Validation Summary

The latest local validation shows that **KMeans volatility clusters** perform better than train-period quantile regimes for this FX volatility classification task.

Best current setup:

- Regime definition: **KMeans volatility clusters**
- Best classical model by Macro F1: **XGBoost**
- Accuracy: **0.6213**
- Balanced accuracy: **0.5786**
- Macro F1: **0.5786**

Why this is safer:

- The train/test split is time-based.
- Quantile thresholds are calculated only from the training period.
- KMeans is fit only on the training-period volatility distribution.
- The test period is only transformed using train-fitted thresholds/clusters, so the evaluation avoids future-data leakage.


## 1. Setup

This notebook downloads real FX data from Yahoo Finance via `yfinance`.

Primary target pair:

- `USDKRW=X` for USD/KRW

Supporting cross-market context:

- `JPYKRW=X`
- `EURKRW=X`
- `SGDKRW=X`

`CNYKRW=X` is not used as a core feature because Yahoo Finance often returns sparse history for that pair.

In [ ]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "csv"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
START_DATE = "2016-01-01"
END_DATE = None

TICKERS = {
    "USDKRW": "USDKRW=X",
    "JPYKRW": "JPYKRW=X",
    "EURKRW": "EURKRW=X",
    "SGDKRW": "SGDKRW=X",
}

MIN_ROWS = 500
CLASS_ORDER = ["low", "normal", "high"]
CLASS_TO_ID = {label: i for i, label in enumerate(CLASS_ORDER)}
ID_TO_CLASS = {i: label for label, i in CLASS_TO_ID.items()}

## 2. Download Yahoo Finance FX Data

Each ticker is downloaded separately so the notebook can gracefully skip sparse or unavailable pairs.

In [ ]:
def download_fx_pair(label: str, ticker: str, start: str = START_DATE, end: str | None = END_DATE) -> pd.DataFrame:
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)

    if df.empty:
        raise ValueError(f"No data returned for {ticker}")

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    keep_cols = [col for col in ["Open", "High", "Low", "Close", "Adj Close"] if col in df.columns]
    df = df[keep_cols].copy()
    df.columns = [f"{label}_{col.lower().replace(' ', '_')}" for col in df.columns]
    df.index.name = "date"
    return df

series = []
available_pairs = []

for label, ticker in TICKERS.items():
    try:
        pair_df = download_fx_pair(label, ticker)
        if len(pair_df) >= MIN_ROWS:
            series.append(pair_df)
            available_pairs.append(label)
            print(f"Loaded {label} ({ticker}): {len(pair_df):,} rows")
        else:
            print(f"Skipped {label} ({ticker}): only {len(pair_df):,} rows")
    except Exception as exc:
        print(f"Skipped {label} ({ticker}): {exc}")

prices = pd.concat(series, axis=1).sort_index()
prices = prices.dropna(subset=["USDKRW_close"])

print("Available pairs:", available_pairs)
print("Price table shape:", prices.shape)
prices.tail()

## 3. Feature Engineering

Features are built from information available up to the observation date.

Feature groups:

- 1-day returns
- rolling volatility over 5, 10, and 20 days
- momentum over 5, 10, and 20 days
- moving-average gap
- USD/KRW intraday range
- cross-currency volatility context

Target:

- next-day USD/KRW absolute return
- converted into `low`, `normal`, and `high` volatility regimes
- compared using two safe target-definition strategies: train-period quantiles and train-fitted KMeans volatility clusters


In [ ]:
def build_fx_features(prices: pd.DataFrame, pairs: list[str]) -> pd.DataFrame:
    df = prices.copy()

    for pair in pairs:
        close = df[f"{pair}_close"]
        df[f"{pair}_return_1d"] = close.pct_change()
        df[f"{pair}_log_return_1d"] = np.log(close).diff()

        for window in [5, 10, 20]:
            df[f"{pair}_vol_{window}d"] = df[f"{pair}_log_return_1d"].rolling(window).std() * np.sqrt(252)
            df[f"{pair}_momentum_{window}d"] = close / close.shift(window) - 1

        df[f"{pair}_ma_gap_5_20"] = close.rolling(5).mean() / close.rolling(20).mean() - 1

    df["USDKRW_intraday_range_pct"] = (df["USDKRW_high"] - df["USDKRW_low"]) / df["USDKRW_close"]
    df["target_next_abs_return"] = df["USDKRW_log_return_1d"].abs().shift(-1)

    feature_cols = [
        col for col in df.columns
        if any(token in col for token in ["return_1d", "vol_", "momentum_", "ma_gap", "intraday_range"])
    ]

    model_df = df[feature_cols + ["target_next_abs_return"]].dropna().copy()
    return model_df

fx_features = build_fx_features(prices, available_pairs)

print("Feature table shape:", fx_features.shape)
print("Date range:", fx_features.index.min(), "to", fx_features.index.max())
fx_features.tail()

## 4. Time-Based Train/Test Split

For time-series data, random split can leak future market behavior into training.  
This notebook uses the first 80% of dates for training and the last 20% for testing.

Two regime-definition methods are compared:

1. **Quantile tercile regime**: low/normal/high thresholds are calculated only from the training period.
2. **KMeans volatility clusters**: KMeans is fit only on the training-period next-day volatility distribution, then applied to the test period.

This keeps the model evaluation safer because no future test distribution is used to fit thresholds or clusters.


In [ ]:
def assign_quantile_regime(value: float, low_threshold: float, high_threshold: float) -> str:
    if value <= low_threshold:
        return "low"
    if value >= high_threshold:
        return "high"
    return "normal"


def build_regime_definitions(base_train: pd.DataFrame, base_test: pd.DataFrame) -> dict[str, dict]:
    definitions = {}

    # Strategy A: train-period quantile labels.
    quantile_train = base_train.copy()
    quantile_test = base_test.copy()
    low_threshold = quantile_train["target_next_abs_return"].quantile(0.33)
    high_threshold = quantile_train["target_next_abs_return"].quantile(0.67)

    for frame in [quantile_train, quantile_test]:
        frame["target_regime"] = frame["target_next_abs_return"].apply(
            lambda x: assign_quantile_regime(x, low_threshold, high_threshold)
        )
        frame["target_id"] = frame["target_regime"].map(CLASS_TO_ID)

    definitions["Quantile tercile"] = {
        "train_df": quantile_train,
        "test_df": quantile_test,
        "method": "train_quantile",
        "low_threshold": low_threshold,
        "high_threshold": high_threshold,
        "cluster_model": None,
        "cluster_centers": None,
    }

    # Strategy B: train-fitted KMeans clusters on next-day absolute USD/KRW return.
    # This avoids leakage because KMeans is fit only on the training-period target distribution.
    cluster_train = base_train.copy()
    cluster_test = base_test.copy()
    kmeans = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=20)
    kmeans.fit(cluster_train[["target_next_abs_return"]])

    centers = kmeans.cluster_centers_.ravel()
    center_order = np.argsort(centers)
    cluster_to_regime = {cluster_id: CLASS_ORDER[position] for position, cluster_id in enumerate(center_order)}

    train_clusters = kmeans.predict(cluster_train[["target_next_abs_return"]])
    test_clusters = kmeans.predict(cluster_test[["target_next_abs_return"]])

    cluster_train["target_regime"] = [cluster_to_regime[x] for x in train_clusters]
    cluster_test["target_regime"] = [cluster_to_regime[x] for x in test_clusters]
    cluster_train["target_id"] = cluster_train["target_regime"].map(CLASS_TO_ID)
    cluster_test["target_id"] = cluster_test["target_regime"].map(CLASS_TO_ID)

    definitions["KMeans volatility clusters"] = {
        "train_df": cluster_train,
        "test_df": cluster_test,
        "method": "train_kmeans_1d_abs_return",
        "low_threshold": None,
        "high_threshold": None,
        "cluster_model": kmeans,
        "cluster_centers": centers,
        "cluster_to_regime": cluster_to_regime,
    }

    return definitions


split_idx = int(len(fx_features) * 0.80)
base_train_df = fx_features.iloc[:split_idx].copy()
base_test_df = fx_features.iloc[split_idx:].copy()

feature_cols = [col for col in fx_features.columns if col != "target_next_abs_return"]
X_train = base_train_df[feature_cols]
X_test = base_test_df[feature_cols]

regime_definitions = build_regime_definitions(base_train_df, base_test_df)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

for name, definition in regime_definitions.items():
    print(f"\n{name}")
    print("Train target mix:")
    print(definition["train_df"]["target_regime"].value_counts(normalize=True).reindex(CLASS_ORDER))
    print("Test target mix:")
    print(definition["test_df"]["target_regime"].value_counts(normalize=True).reindex(CLASS_ORDER))
    if definition["method"] == "train_quantile":
        print("low <=", definition["low_threshold"])
        print("high >=", definition["high_threshold"])
    else:
        centers = pd.Series(definition["cluster_centers"]).sort_values().to_list()
        print("ordered cluster centers:", centers)

## 5. Model Comparison

This notebook compares two things separately:

1. **Regime definition method**
   - train-period quantile terciles
   - train-fitted KMeans volatility clusters

2. **Prediction model**
   - Logistic Regression baseline
   - Random Forest
   - LightGBM
   - XGBoost
   - scikit-learn HistGradientBoostingClassifier

Logistic Regression is used as the baseline model because it is simple, explainable, and useful as a sanity check.

**Current result:** KMeans volatility clusters produce the stronger target definition, and XGBoost is the best classical model by Macro F1.

**Note:** Gradient boosting is not newer than XGBoost. Gradient boosting is the algorithm family. XGBoost and LightGBM are optimized gradient-boosting implementations, while `HistGradientBoostingClassifier` is scikit-learn's modern histogram-based implementation.


In [ ]:
def make_model_specs(y_train: pd.Series, y_train_id: np.ndarray) -> dict[str, dict]:
    return {
        "Logistic Regression baseline": {
            "model": Pipeline(
                steps=[
                    ("scaler", StandardScaler()),
                    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
                ]
            ),
            "target": y_train,
            "predict_labels": True,
        },
        "Random Forest": {
            "model": RandomForestClassifier(
                n_estimators=300,
                max_depth=8,
                min_samples_leaf=20,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
            "target": y_train,
            "predict_labels": True,
        },
        "LightGBM": {
            "model": LGBMClassifier(
                n_estimators=300,
                learning_rate=0.03,
                num_leaves=15,
                subsample=0.85,
                colsample_bytree=0.85,
                objective="multiclass",
                random_state=RANDOM_STATE,
                verbose=-1,
            ),
            "target": y_train_id,
            "predict_labels": False,
        },
        "XGBoost": {
            "model": XGBClassifier(
                n_estimators=300,
                learning_rate=0.03,
                max_depth=4,
                subsample=0.85,
                colsample_bytree=0.85,
                objective="multi:softprob",
                eval_metric="mlogloss",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
            "target": y_train_id,
            "predict_labels": False,
        },
        "Sklearn HistGradientBoosting": {
            "model": HistGradientBoostingClassifier(
                max_iter=250,
                learning_rate=0.04,
                max_leaf_nodes=15,
                l2_regularization=0.05,
                random_state=RANDOM_STATE,
            ),
            "target": y_train,
            "predict_labels": True,
        },
    }


results = []
fitted_models = {}

for regime_name, definition in regime_definitions.items():
    y_train = definition["train_df"]["target_regime"]
    y_test = definition["test_df"]["target_regime"]

    label_encoder = LabelEncoder()
    y_train_id = label_encoder.fit_transform(y_train)

    model_specs = make_model_specs(y_train, y_train_id)

    for model_name, config in model_specs.items():
        model = config["model"]
        model.fit(X_train, config["target"])
        raw_pred = model.predict(X_test)

        if config["predict_labels"]:
            y_pred_model = raw_pred
        else:
            y_pred_model = label_encoder.inverse_transform(raw_pred.astype(int))

        fitted_models[(regime_name, model_name)] = {
            "model": model,
            "predict_labels": config["predict_labels"],
            "label_encoder": label_encoder,
            "y_train": y_train,
            "y_test": y_test,
        }

        results.append(
            {
                "regime_definition": regime_name,
                "model": model_name,
                "accuracy": accuracy_score(y_test, y_pred_model),
                "balanced_accuracy": balanced_accuracy_score(y_test, y_pred_model),
                "precision_macro": precision_score(y_test, y_pred_model, average="macro", zero_division=0),
                "recall_macro": recall_score(y_test, y_pred_model, average="macro", zero_division=0),
                "f1_macro": f1_score(y_test, y_pred_model, average="macro", zero_division=0),
            }
        )

metrics_df = pd.DataFrame(results).sort_values(["f1_macro", "balanced_accuracy"], ascending=False)
metrics_df

## 6. Choose Final Model and Inspect Performance

The final model is selected by macro F1, excluding the majority-class baseline.

In [ ]:
best_row = metrics_df.sort_values(["f1_macro", "balanced_accuracy"], ascending=False).iloc[0]
best_regime_definition = best_row["regime_definition"]
best_model_name = best_row["model"]
best_model_config = fitted_models[(best_regime_definition, best_model_name)]
best_model = best_model_config["model"]
label_encoder = best_model_config["label_encoder"]
y_train = best_model_config["y_train"]
y_test = best_model_config["y_test"]
train_df = regime_definitions[best_regime_definition]["train_df"]
test_df = regime_definitions[best_regime_definition]["test_df"]


def compact_regime_metadata(definition: dict) -> dict:
    cluster_centers = definition.get("cluster_centers")
    cluster_to_regime = definition.get("cluster_to_regime")

    cluster_centers_list = None if cluster_centers is None else [float(x) for x in cluster_centers]

    return {
        "method": definition["method"],
        "low_threshold": None if definition.get("low_threshold") is None else float(definition["low_threshold"]),
        "high_threshold": None if definition.get("high_threshold") is None else float(definition["high_threshold"]),
        "cluster_centers": cluster_centers_list,
        "cluster_centers_sorted": None if cluster_centers_list is None else sorted(cluster_centers_list),
        "cluster_to_regime": None if cluster_to_regime is None else {int(k): str(v) for k, v in cluster_to_regime.items()},
    }


regime_metadata_by_definition = {
    name: compact_regime_metadata(definition)
    for name, definition in regime_definitions.items()
}
best_regime_metadata = regime_metadata_by_definition[best_regime_definition]

print("Best regime definition:", best_regime_definition)
print("Best model:", best_model_name)
print("Selection metric: highest macro F1, then balanced accuracy as tie-breaker")

raw_pred = best_model.predict(X_test)
if best_model_config["predict_labels"]:
    y_pred = raw_pred
else:
    y_pred = label_encoder.inverse_transform(raw_pred.astype(int))

print(classification_report(y_test, y_pred, labels=CLASS_ORDER, zero_division=0))

confusion = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=CLASS_ORDER),
    index=[f"Actual {label}" for label in CLASS_ORDER],
    columns=[f"Predicted {label}" for label in CLASS_ORDER],
)
confusion


## 7. Feature Importance / Signal Strength

Tree models provide direct feature importance.  
For logistic regression, coefficient magnitude is used as a simple signal-strength proxy.

In [ ]:
def get_signal_strength(model, feature_names: list[str]) -> pd.DataFrame:
    estimator = model.named_steps["model"] if isinstance(model, Pipeline) else model

    if hasattr(estimator, "feature_importances_"):
        values = estimator.feature_importances_
    elif hasattr(estimator, "coef_"):
        values = np.abs(estimator.coef_).mean(axis=0)
    else:
        values = np.zeros(len(feature_names))

    return (
        pd.DataFrame({"feature": feature_names, "importance": values})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

importance_df = get_signal_strength(best_model, feature_cols)
importance_df.head(15)


## 8. Prediction Output and Operational Recommendation

Predictions are converted into a simple operating recommendation:

- **low:** normal monitoring
- **normal:** standard readiness
- **high:** prepare rate-change messaging, transaction monitoring, and support readiness

In [ ]:
def recommendation_for_regime(regime: str) -> str:
    if regime == "high":
        return "Prepare rate-change messaging, transaction monitoring, and support readiness."
    if regime == "normal":
        return "Use standard monitoring and normal lifecycle messaging."
    return "Normal operations; consider using calm-market windows for product education."

if hasattr(best_model, "predict_proba"):
    proba = best_model.predict_proba(X_test)
    raw_classes = list(best_model.classes_)
    if best_model_config["predict_labels"]:
        proba_classes = raw_classes
    else:
        proba_classes = list(label_encoder.inverse_transform(np.array(raw_classes).astype(int)))
else:
    proba = np.zeros((len(X_test), len(CLASS_ORDER)))
    proba_classes = CLASS_ORDER

prediction_df = pd.DataFrame(index=X_test.index)
prediction_df.index.name = "date"
prediction_df["actual_regime"] = y_test.values
prediction_df["predicted_regime"] = y_pred

for label in CLASS_ORDER:
    if label in proba_classes:
        prediction_df[f"prob_{label}"] = proba[:, proba_classes.index(label)]
    else:
        prediction_df[f"prob_{label}"] = np.nan

prediction_df["recommended_action"] = prediction_df["predicted_regime"].map(recommendation_for_regime)
prediction_df.tail(10)

## 9. Export Dashboard-Ready Outputs

These files can later be added to the Streamlit FX Market Intelligence page.

In [ ]:
features_export = fx_features.copy()
features_export.index.name = "date"
features_export = features_export.reset_index()

metrics_export = metrics_df.copy()
metrics_export["selected_model_flag"] = ((metrics_export["regime_definition"].eq(best_regime_definition)) & (metrics_export["model"].eq(best_model_name))).astype(int)

predictions_export = prediction_df.reset_index()
importance_export = importance_df.copy()
confusion_export = confusion.reset_index().rename(columns={"index": "actual_regime"})
best_by_f1 = metrics_df.sort_values(["regime_definition", "f1_macro", "balanced_accuracy"], ascending=[True, False, False]).drop_duplicates("regime_definition")
best_by_balanced_accuracy = metrics_df.sort_values(["regime_definition", "balanced_accuracy", "f1_macro"], ascending=[True, False, False]).drop_duplicates("regime_definition")

regime_comparison_export = best_by_f1[["regime_definition", "model", "f1_macro", "balanced_accuracy"]].rename(
    columns={
        "model": "best_model_by_f1",
        "f1_macro": "best_f1_macro",
        "balanced_accuracy": "balanced_accuracy_for_best_f1_model",
    }
).merge(
    best_by_balanced_accuracy[["regime_definition", "model", "balanced_accuracy", "f1_macro"]].rename(
        columns={
            "model": "best_model_by_balanced_accuracy",
            "balanced_accuracy": "best_balanced_accuracy",
            "f1_macro": "f1_macro_for_best_balanced_accuracy_model",
        }
    ),
    on="regime_definition",
    how="left",
)
regime_metadata_export = pd.DataFrame(
    [
        {
            "regime_definition": name,
            "method": metadata["method"],
            "low_threshold": metadata["low_threshold"],
            "high_threshold": metadata["high_threshold"],
            "cluster_centers": None if metadata["cluster_centers"] is None else ", ".join(f"{x:.8f}" for x in metadata["cluster_centers"]),
            "cluster_centers_sorted": None if metadata["cluster_centers_sorted"] is None else ", ".join(f"{x:.8f}" for x in metadata["cluster_centers_sorted"]),
            "cluster_to_regime": None if metadata["cluster_to_regime"] is None else str(metadata["cluster_to_regime"]),
        }
        for name, metadata in regime_metadata_by_definition.items()
    ]
)

features_export.to_csv(OUTPUT_DIR / "fx_market_features.csv", index=False)
metrics_export.to_csv(OUTPUT_DIR / "fx_volatility_model_metrics.csv", index=False)
predictions_export.to_csv(OUTPUT_DIR / "fx_volatility_predictions.csv", index=False)
importance_export.to_csv(OUTPUT_DIR / "fx_volatility_feature_importance.csv", index=False)
confusion_export.to_csv(OUTPUT_DIR / "fx_volatility_confusion_matrix.csv", index=False)
regime_comparison_export.to_csv(OUTPUT_DIR / "fx_volatility_regime_definition_comparison.csv", index=False)
regime_metadata_export.to_csv(OUTPUT_DIR / "fx_volatility_regime_metadata.csv", index=False)

joblib.dump(
    {
        "model_name": best_model_name,
        "model": best_model,
        "feature_cols": feature_cols,
        "class_order": CLASS_ORDER,
        "regime_definition": best_regime_definition,
        "regime_metadata": best_regime_metadata,
        "available_pairs": available_pairs,
    },
    MODEL_DIR / "fx_volatility_regime_model.pkl",
)

print("Exported:")
for file_name in [
    "fx_market_features.csv",
    "fx_volatility_model_metrics.csv",
    "fx_volatility_predictions.csv",
    "fx_volatility_feature_importance.csv",
    "fx_volatility_confusion_matrix.csv",
    "fx_volatility_regime_definition_comparison.csv",
    "fx_volatility_regime_metadata.csv",
]:
    path = OUTPUT_DIR / file_name
    print(file_name, f"{path.stat().st_size:,} bytes")
print("Saved model:", MODEL_DIR / "fx_volatility_regime_model.pkl")


## 10. Latest Market Signal

This gives a portfolio-friendly product output: a current operational signal rather than a raw technical forecast.

In [ ]:
latest_X = fx_features[feature_cols].tail(1)
latest_date = latest_X.index[0]
latest_raw_pred = best_model.predict(latest_X)[0]
latest_regime = latest_raw_pred if best_model_config["predict_labels"] else label_encoder.inverse_transform([int(latest_raw_pred)])[0]

if hasattr(best_model, "predict_proba"):
    latest_proba = best_model.predict_proba(latest_X)[0]
    raw_classes = list(best_model.classes_)
    if best_model_config["predict_labels"]:
        latest_classes = raw_classes
    else:
        latest_classes = list(label_encoder.inverse_transform(np.array(raw_classes).astype(int)))
    latest_proba_map = dict(zip(latest_classes, latest_proba))
else:
    latest_proba_map = {}

print("Latest observation date:", latest_date.date())
print("Predicted next-day volatility regime:", latest_regime)
print("Recommended action:", recommendation_for_regime(latest_regime))
print("Probabilities:")
for label in CLASS_ORDER:
    print(label, round(float(latest_proba_map.get(label, np.nan)), 3))


In [ ]:
deep_learning_results = []


## 11. Optional PyTorch Experiment

This section is optional.  
It trains a small MLP classifier on the same engineered features.  
For this portfolio, the scikit-learn model remains the main model because it is easier to explain to product and analytics stakeholders.

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from sklearn.preprocessing import LabelEncoder

    torch.manual_seed(RANDOM_STATE)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype("float32")
    X_test_scaled = scaler.transform(X_test).astype("float32")

    label_encoder = LabelEncoder()
    y_train_id = label_encoder.fit_transform(y_train).astype("int64")
    y_test_id = label_encoder.transform(y_test).astype("int64")

    X_train_tensor = torch.tensor(X_train_scaled)
    y_train_tensor = torch.tensor(y_train_id)
    X_test_tensor = torch.tensor(X_test_scaled)

    class FXRegimeMLP(nn.Module):
        def __init__(self, input_dim: int, output_dim: int):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 64),
                nn.ReLU(),
                nn.Dropout(0.20),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, output_dim),
            )

        def forward(self, x):
            return self.net(x)

    torch_model = FXRegimeMLP(X_train_scaled.shape[1], len(label_encoder.classes_))
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(torch_model.parameters(), lr=0.001)

    for epoch in range(40):
        torch_model.train()
        optimizer.zero_grad()
        logits = torch_model(X_train_tensor)
        loss = criterion(logits, y_train_tensor)
        loss.backward()
        optimizer.step()

    torch_model.eval()
    with torch.no_grad():
        test_logits = torch_model(X_test_tensor)
        torch_pred_id = test_logits.argmax(dim=1).numpy()

    torch_pred = label_encoder.inverse_transform(torch_pred_id)
    torch_metrics = {
        "model": "PyTorch MLP",
        "accuracy": accuracy_score(y_test, torch_pred),
        "precision_macro": precision_score(y_test, torch_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, torch_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, torch_pred, average="macro", zero_division=0),
    }
    deep_learning_results.append(torch_metrics)
    print("PyTorch MLP macro F1:", torch_metrics["f1_macro"])
    print(classification_report(y_test, torch_pred, labels=CLASS_ORDER, zero_division=0))

except ModuleNotFoundError:
    print("PyTorch is not installed. Install it from requirements-ml.txt if you want to run this section.")

## 12. Optional TensorFlow Experiment

This section mirrors the PyTorch experiment using Keras.  
It is included to show flexibility, but it is not required for the Streamlit dashboard.

In [ ]:
try:
    import tensorflow as tf
    from sklearn.preprocessing import LabelEncoder

    tf.random.set_seed(RANDOM_STATE)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype("float32")
    X_test_scaled = scaler.transform(X_test).astype("float32")

    label_encoder = LabelEncoder()
    y_train_id = label_encoder.fit_transform(y_train)
    y_test_id = label_encoder.transform(y_test)

    tf_model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dropout(0.20),
            tf.keras.layers.Dense(32, activation="relu"),
            tf.keras.layers.Dense(len(label_encoder.classes_), activation="softmax"),
        ]
    )
    tf_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    tf_model.fit(X_train_scaled, y_train_id, epochs=30, batch_size=64, verbose=0)

    tf_pred_id = tf_model.predict(X_test_scaled, verbose=0).argmax(axis=1)
    tf_pred = label_encoder.inverse_transform(tf_pred_id)

    tf_metrics = {
        "model": "TensorFlow MLP",
        "accuracy": accuracy_score(y_test, tf_pred),
        "precision_macro": precision_score(y_test, tf_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, tf_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, tf_pred, average="macro", zero_division=0),
    }
    deep_learning_results.append(tf_metrics)
    deep_learning_metrics_df = pd.DataFrame(deep_learning_results)
    deep_learning_metrics_df.to_csv(OUTPUT_DIR / "fx_volatility_deep_learning_metrics.csv", index=False)
    print("TensorFlow MLP macro F1:", tf_metrics["f1_macro"])
    print(classification_report(y_test, tf_pred, labels=CLASS_ORDER, zero_division=0))
    deep_learning_metrics_df

except ModuleNotFoundError:
    print("TensorFlow is not installed. Install it from requirements-ml.txt if you want to run this section.")

## Portfolio Interpretation

This module strengthens the FX fintech story without overclaiming exact exchange-rate forecasting.

Recommended wording for the portfolio:

> I built a real-market FX volatility regime prediction module using Yahoo Finance data. The model classifies next-day USD/KRW volatility into low, normal, and high regimes, providing an operational signal for transaction monitoring, rate-change messaging, and support readiness.

This aligns with Switchwon's flex role around AI/financial time-series support while keeping the main portfolio focused on product analytics, retention, cohort, KPI dashboards, and data marts.